In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models, transforms
from torchvision.datasets import ImageFolder

from torch.utils.data import DataLoader, random_split

from pathlib import Path

In [3]:
BASE_DIR = Path(r"C:\Users\bhall\OneDrive\Desktop\project\data")

color_dir = BASE_DIR / "color"

print(color_dir)
print(color_dir.exists())

C:\Users\bhall\OneDrive\Desktop\project\data\color
True


In [4]:

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [6]:
test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [7]:
dataset = ImageFolder(
    root=color_dir,
    transform=test_transform
)

print("Total Images :", len(dataset))
print("Total Classes:", len(dataset.classes))

Total Images : 54316
Total Classes: 38


In [8]:
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size]
)

print("Train :", len(train_dataset))
print("Validation :", len(val_dataset))
print("Test :", len(test_dataset))

Train : 38021
Validation : 8147
Test : 8148


In [9]:
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Test batches:", len(test_loader))

Test batches: 255


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mobilenet = models.mobilenet_v2(weights=None)

mobilenet.classifier[1] = nn.Linear(1280, 38)

model_path = Path(r"C:\Users\bhall\OneDrive\Desktop\project\models\mobilenet_plant_disease_new.pth")

mobilenet.load_state_dict(torch.load(model_path, map_location=device))

mobilenet = mobilenet.to(device)
mobilenet.eval()

print("Model loaded successfully!")
print("Device:", device)

Model loaded successfully!
Device: cpu


In [11]:
criterion = nn.CrossEntropyLoss()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = mobilenet(images)

        loss = criterion(outputs, labels)

        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_loss = test_loss / len(test_loader)
test_accuracy = 100 * correct / total

print("=================================")
print("Test Results")
print("=================================")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.2f}%")

Test Results
Test Loss     : 0.0769
Test Accuracy : 97.68%
